# Self-Consistency 自一致性 - 详细教程

## 学习目标
1. 理解自一致性的原理
2. 掌握投票策略
3. 学会实际应用

## 目录
1. [什么是 Self-Consistency](#1-什么是-self-consistency)
2. [数学原理](#2-数学原理)
3. [核心数据结构](#3-核心数据结构)
4. [投票策略](#4-投票策略)
5. [SelfConsistency 类](#5-selfconsistency-类)
6. [实战案例](#6-实战案例)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.self_consistency import (
    SampledPath, ConsistencyResult,
    VotingStrategy, MajorityVoting, WeightedVoting,
    SelfConsistency
)
print("模块加载成功！")

---
## 1. 什么是 Self-Consistency

### 1.1 核心思想

In [ ]:
print("""
Self-Consistency 核心思想：

┌─────────────────────────────────────────────────────┐
│                                                     │
│    问题 ──┬──> 推理路径1 ──> 答案A                   │
│           ├──> 推理路径2 ──> 答案A                   │
│           ├──> 推理路径3 ──> 答案B                   │
│           ├──> 推理路径4 ──> 答案A                   │
│           └──> 推理路径5 ──> 答案A                   │
│                                                     │
│    投票结果: A(4票) > B(1票)                         │
│    最终答案: A                                       │
│                                                     │
└─────────────────────────────────────────────────────┘

关键洞察：正确答案更可能被多条推理路径发现
""")

### 1.2 为什么有效

In [ ]:
print("""
Self-Consistency 有效的原因：

1. 多样性采样
   - 使用较高温度生成多条路径
   - 不同路径可能发现不同解法

2. 错误过滤
   - 错误推理通常不一致
   - 正确推理更可能收敛

3. 统计优势
   - 多数投票减少随机错误
   - 类似集成学习的效果
""")

---
## 2. 数学原理

### 2.1 多数投票

In [ ]:
print("""
多数投票公式：

a* = argmax_a Σ 1[aᵢ = a]

其中：
  - a* 是最终答案
  - aᵢ 是第i条路径的答案
  - 1[·] 是指示函数

示例：
  路径1: 答案=12
  路径2: 答案=12
  路径3: 答案=10
  路径4: 答案=12
  
  投票: 12(3票), 10(1票)
  结果: 12
""")

### 2.2 加权投票

In [ ]:
print("""
加权投票公式：

a* = argmax_a Σ cᵢ · 1[aᵢ = a]

其中 cᵢ 是第i条路径的置信度

示例：
  路径1: 答案=12, 置信度=0.9
  路径2: 答案=12, 置信度=0.8
  路径3: 答案=10, 置信度=0.3
  
  加权: 12(1.7), 10(0.3)
  结果: 12
""")

---
## 3. 核心数据结构

### 3.1 SampledPath 采样路径

In [ ]:
# 创建采样路径
path1 = SampledPath(
    reasoning="15% = 0.15, 0.15 × 80 = 12",
    answer="12",
    confidence=0.9
)

path2 = SampledPath(
    reasoning="15/100 × 80 = 1200/100 = 12",
    answer="12",
    confidence=0.85
)

path3 = SampledPath(
    reasoning="大约是10左右",
    answer="10",
    confidence=0.3
)

paths = [path1, path2, path3]

print("采样路径：")
for i, p in enumerate(paths, 1):
    print(f"  {i}. 答案={p.answer}, 置信度={p.confidence}")
    print(f"     推理: {p.reasoning}")

### 3.2 ConsistencyResult 结果

In [ ]:
from collections import Counter

# 创建结果
vote_counts = Counter(p.answer for p in paths)
result = ConsistencyResult(
    question="15%的80是多少？",
    final_answer="12",
    confidence=0.8,
    vote_counts=dict(vote_counts),
    paths=paths,
    agreement_ratio=2/3
)

print("一致性结果：")
print(f"  问题: {result.question}")
print(f"  答案: {result.final_answer}")
print(f"  投票: {result.vote_counts}")
print(f"  一致率: {result.agreement_ratio:.0%}")

---
## 4. 投票策略

### 4.1 多数投票

In [ ]:
majority = MajorityVoting()
winner, conf = majority.vote(paths)

print(f"多数投票：")
print(f"  获胜: {winner}")
print(f"  置信度: {conf:.0%}")

### 4.2 加权投票

In [ ]:
weighted = WeightedVoting()
winner, conf = weighted.vote(paths)

print(f"加权投票：")
print(f"  获胜: {winner}")
print(f"  置信度: {conf:.0%}")

---
## 5. SelfConsistency 类

In [ ]:
# 创建实例
sc = SelfConsistency(
    n_samples=5,
    voting_strategy=MajorityVoting(),
    temperature=0.7
)

print(f"SelfConsistency 配置：")
print(f"  采样数: {sc._n_samples}")
print(f"  温度: {sc._temperature}")

---
## 6. 实战案例

In [ ]:
# 模拟多路径采样
print("实战案例：数学题")
print("="*40)
print("问题：一个班有30人，男生比女生多6人，男生有多少人？")
print("\n采样结果：")

sample_paths = [
    SampledPath("设女生x，男生x+6，2x+6=30，x=12，男生18", "18", 0.9),
    SampledPath("(30+6)/2=18", "18", 0.85),
    SampledPath("30/2+3=18", "18", 0.8),
    SampledPath("30-6=24，24/2=12", "12", 0.4),
    SampledPath("男生比女生多6，所以18人", "18", 0.7),
]

for i, p in enumerate(sample_paths, 1):
    print(f"  路径{i}: {p.answer} (置信度:{p.confidence})")

# 投票
winner, conf = MajorityVoting().vote(sample_paths)
print(f"\n最终答案: {winner} (一致率: {conf:.0%})")

---
## 最佳实践

In [ ]:
print("""
Self-Consistency 最佳实践：

1. 采样数量
   - 简单问题: 3-5次
   - 复杂问题: 10-20次
   - 更多不一定更好

2. 温度设置
   - 0.5-0.8 通常最佳
   - 太低: 路径相似
   - 太高: 质量下降

3. 投票策略
   - 多数投票: 简单可靠
   - 加权投票: 考虑置信度

4. 一致性阈值
   - >80%: 高置信
   - 50-80%: 中等
   - <50%: 需要人工检查
""")